# Preferred Feature Images (PFI) via reverse correlation

Ported from `Run_PFI.m`'s reverse-correlation section in
[cogilab/Face](https://github.com/cogilab/Face) (Baek et al., "Face detection
in untrained deep neural networks," Nature Communications, 2021) -- the
official code for the paper. Only the reverse-correlation method is ported
(not the XDream/GAN variant, which needs an external pretrained GAN).

**Algorithm (unchanged from the paper, image size adapted for CIFAR-native
32x32 instead of AlexNet's 227x227):**
1. Start PFI at mid-gray.
2. Each iteration, add many small Gaussian "dot" perturbations (both
   polarities) at a grid of positions to the current PFI, forming a batch of
   probe images.
3. Forward-pass all probes through the network (weights frozen, no
   backprop/gradient ascent at all).
4. Take the response (raw logit) at the target unit for each probe.
5. Update PFI to the response-weighted average of the probe images
   (= reverse correlation), amplify the step by 10x (matches the paper).
6. Repeat for 100 iterations.

Target unit here: **a specific output class's logit** (e.g. "what does this
network's 'cat' output prefer to see"), on a checkpoint from the
`cifarnative_gradual_lr.ipynb` (g, l) sweep.

Adapted parameters (paper's own, scaled from 227px to 32px): grid of dot
positions reduced from 50x50 to 16x16, dot_size scaled down proportionally.
Iteration count (100) and the x10 update amplification are unchanged.

In [ ]:
%matplotlib inline
import sys, os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

sys.path.append('..')
from src.models.resnet import resnet18  # CIFAR-native (3x3 stride-1 stem, 32x32)
from custom.figure import mm, color

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams['font.family'] = 'DejaVu Sans'

output_size = 10
classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

CIFAR_MEAN = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1).to(device)
CIFAR_STD = torch.tensor((0.2023, 0.1994, 0.2010)).view(1, 3, 1, 1).to(device)

def normalize(x):
    """x: (N, 3, H, W) in [0, 1] pixel space -> normalized model input."""
    return (x - CIFAR_MEAN) / CIFAR_STD

figure_dir = os.path.join("..", "figures", "pfi")
os.makedirs(figure_dir, exist_ok=True)


## Discover checkpoints, average PFI over num_net

Instead of one hardcoded net, find every saved `best_model_{tag}_{i}.pth` in a results dir, group by tag, and average the PFI across all available nets for that tag -- more robust than a single net's snapshot.

In [ ]:
import re

def load_model(ckpt_path):
    model = resnet18(num_classes=output_size).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model

def discover_checkpoints(results_dir):
    """Group best_model_{tag}_{net_idx}.pth files in results_dir by tag.
    Returns {tag: {net_idx: path}}."""
    pattern = re.compile(r"^best_model_(.+)_(\d+)\.pth$")
    groups = {}
    for fname in os.listdir(results_dir):
        m = pattern.match(fname)
        if m:
            tag, net_idx = m.group(1), int(m.group(2))
            groups.setdefault(tag, {})[net_idx] = os.path.join(results_dir, fname)
    return groups

gl_results_dir = os.path.join("..", "results", "cifar10-native-gradual-lr")
warmup_results_dir = os.path.join("..", "results", "cifar10-native-0919")

gl_checkpoints = discover_checkpoints(gl_results_dir)
print(f"found {len(gl_checkpoints)} tags in {gl_results_dir}:")
for tag, nets in sorted(gl_checkpoints.items()):
    print(f"  {tag}: {len(nets)} nets")

# example single model, for the quick single-class/all-classes demo cells below
model = load_model(gl_checkpoints["gradual_r10_g20l30"][0])


## Reverse-correlation PFI (ported from `Run_PFI.m`)

In [ ]:
def compute_pfi(model, target_class, img_size=32, grid_n=16, dot_size=2.0,
                 iterations=100, amplify=10.0, seed=0, return_history=False):
    """Reverse-correlation Preferred Feature Image for a single output class
    logit. Network weights are frozen throughout -- this is pure
    response-weighted averaging over small random probes, not gradient
    ascent (matches Run_PFI.m's reverse-correlation section exactly)."""
    rng = np.random.RandomState(seed)

    xs = np.linspace(dot_size, img_size - dot_size, grid_n)
    ys = np.linspace(dot_size, img_size - dot_size, grid_n)
    xx, yy = np.meshgrid(xs, ys)
    positions = np.stack([xx.ravel(), yy.ravel()], axis=1)

    yv, xv = np.meshgrid(np.arange(img_size), np.arange(img_size), indexing="ij")
    dots = []
    for px, py in positions:
        d2 = (xv - px) ** 2 + (yv - py) ** 2
        g = np.exp(-d2 / (2 * dot_size ** 2)) * 0.5
        dots.append(g)
    dots = np.stack(dots, axis=0)                      # (grid_n^2, H, W)
    dots = np.repeat(dots[:, None, :, :], 3, axis=1)    # (grid_n^2, 3, H, W)
    dots = np.concatenate([-dots, dots], axis=0)        # both polarities
    dots = torch.from_numpy(dots).float().to(device)

    pfi = torch.full((1, 3, img_size, img_size), 0.5, device=device)  # mid-gray, [0,1]
    history = [pfi.squeeze(0).cpu().clone()] if return_history else None

    with torch.no_grad():
        for it in range(iterations):
            pfi_prev = pfi.clone()

            imgs = (pfi + dots).clamp(0, 1)             # (N, 3, H, W)
            outputs = model(normalize(imgs))
            resp = outputs[:, target_class]              # raw logit, target unit

            weight = resp - resp.min()
            weighted = (imgs * weight.view(-1, 1, 1, 1)).sum(dim=0, keepdim=True) / weight.sum()

            diff = weighted - pfi_prev
            pfi = (pfi_prev + diff * amplify).clamp(0, 1)

            if return_history:
                history.append(pfi.squeeze(0).cpu().clone())

    result = pfi.squeeze(0).cpu()
    return (result, history) if return_history else result


def compute_pfi_averaged(ckpts_by_net, target_class, **kwargs):
    """Average the PFI across every net_idx available for one tag."""
    pfis = []
    for net_idx in sorted(ckpts_by_net):
        m = load_model(ckpts_by_net[net_idx])
        pfis.append(compute_pfi(m, target_class, **kwargs))
    return torch.stack(pfis, dim=0).mean(dim=0)


Run for one class and show the result.

In [ ]:
target_class = classes.index("cat")

pfi_img, history = compute_pfi(model, target_class, return_history=True)

plt.figure(figsize=(30 * mm, 30 * mm))
plt.imshow(pfi_img.permute(1, 2, 0).numpy())
plt.title(f"PFI: {classes[target_class]}")
plt.axis("off")
plt.savefig(os.path.join(figure_dir, f"pfi_{classes[target_class]}.svg"))
plt.show()


All 10 classes side by side, for this one checkpoint.

In [ ]:
fig, axes = plt.subplots(1, output_size, figsize=(output_size * 1.8, 2.2))
for c in range(output_size):
    pfi_c = compute_pfi(model, c)
    axes[c].imshow(pfi_c.permute(1, 2, 0).numpy())
    axes[c].set_title(classes[c], fontsize=8)
    axes[c].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "pfi_all_classes.svg"))
plt.show()


## Compare across ALL discovered training conditions

Same target class, one averaged-over-nets PFI per tag, for every tag found under `gl_results_dir` -- no hardcoded checkpoint list.

In [ ]:
target_class = classes.index("cat")

tags = sorted(gl_checkpoints.keys())
pfi_by_tag = {tag: compute_pfi_averaged(gl_checkpoints[tag], target_class) for tag in tags}

ncols = 6
nrows = int(np.ceil(len(tags) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.8, nrows * 2))
axes = np.atleast_2d(axes)

for i, tag in enumerate(tags):
    ax = axes[i // ncols, i % ncols]
    ax.imshow(pfi_by_tag[tag].permute(1, 2, 0).numpy())
    ax.set_title(tag.replace("gradual_r10_", ""), fontsize=7)
    ax.axis("off")

for i in range(len(tags), nrows * ncols):
    axes[i // ncols, i % ncols].axis("off")

plt.suptitle(f"PFI for '{classes[target_class]}', averaged over available nets per condition", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"pfi_{classes[target_class]}_all_conditions.svg"))
plt.show()
